## tavily로 뉴스 기사 크롤링 후 Neo4j에 임베딩까지

In [23]:
import os
import getpass
from dotenv import load_dotenv
from tavily import TavilyClient

os.environ["TAVILY_API_KEY"] = "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"

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

Tavily-langchain 통합을 통해 다음과 같은 모듈형 도구를 정의한다.
1. **Search** the web for relevant information
2. **Extract** content from specific web pages
3. **Crawl** entire websites

In [24]:
# Define the set of web tools our agent will use to interact with the Tavily API
from langchain_tavily import TavilySearch, TavilyExtract, TavilyCrawl

# Define the langchain search tool
search = TavilySearch(max_results=10, topic="general")

# Define the langchain Extract tool
extract = TavilyExtract(extract_depth="advanced")

# Define the langchain search tool
crawl = TavilyCrawl()

In [25]:
# Instantiate the OpenAI foundation models
from langchain_openai import ChatOpenAI

# gpt-5-nano
gpt_5_nano = ChatOpenAI(model="gpt-5-nano")

### Web Agent
아래에서는 Tavily를 사용한 웹 에이전트를 설계한다. 이는 LLM, 웹 도구, 시스템 프롬프트, 총 3개로 구성된다.<br>
언어모델은 에이전트의 뇌를 담당하고 도구들은 에이전트가 인터넷과 상호작용하여 정보를 얻도록 한다.<br>
시스템 프롬프트는 에이전트의 행동, 이유, 도구 사용 시점 등에 대한 가이드를 제공한다.

In [26]:
import datetime
from langchain.agents import create_agent
from langchain.messages import SystemMessage
from langchain_core.prompts import MessagesPlaceholder


today = datetime.datetime.today().strftime("%A, %B %d, %Y")

web_agent = create_agent(
    model="gpt-5-nano",
    tools=[search, extract, crawl],
    system_prompt="""    
You are an information extraction and knowledge graph construction agent.

Your task is to discover and extract **relationships between companies** using web search results provided by the Tavily Search tool and convert them into **Neo4j graph database Cypher queries**.

You must strictly follow the ontology and output format rules below.

---

# Objective

Build a **company relationship graph** where:

* Nodes represent companies
* Edges represent relationships between companies
* The graph will be stored in Neo4j

You must extract factual company relationships from web sources and convert them into Cypher queries.

---

# Node Definition

All nodes must be of type `Company`.

Example node:

(:Company {name: "Samsung Electronics"})

Only create nodes for companies.

Do NOT create nodes for people, products, events, governments, or technologies.

---

# Allowed Relationship Types

You may only use the following edge types.

SUPPLIES
Meaning: Company A supplies products or components to Company B.

CUSTOMER_OF
Meaning: Company A is a customer of Company B.

PARTNERS_WITH
Meaning: Strategic partnership, collaboration, joint development.

COMPETES_WITH
Meaning: Two companies compete in the same market.

OWNS
Meaning: Ownership or controlling stake.

INVESTS_IN
Meaning: Investment relationship without full control.

LICENSES_TECH_TO
Meaning: Technology or IP licensing.

DEPENDS_ON
Meaning: Critical supply or operational dependency.

---

# Relationship Direction

SUPPLIES
(A)-[:SUPPLIES]->(B)

CUSTOMER_OF
(A)-[:CUSTOMER_OF]->(B)

PARTNERS_WITH
(A)-[:PARTNERS_WITH]->(B)

COMPETES_WITH
(A)-[:COMPETES_WITH]->(B)

OWNS
(A)-[:OWNS]->(B)

INVESTS_IN
(A)-[:INVESTS_IN]->(B)

LICENSES_TECH_TO
(A)-[:LICENSES_TECH_TO]->(B)

DEPENDS_ON
(A)-[:DEPENDS_ON]->(B)

---

# Edge Properties

Each relationship must include the following properties:

source
URL of the source article

confidence
A value between 0 and 1 indicating extraction confidence

Example:

[:SUPPLIES {
source: "https://example.com/article",
confidence: 0.84
}]

---

# Neo4j Cypher Output Format

You must output **only Cypher queries**.

Use the following template.

MERGE (a:Company {name: "Company A"})
MERGE (b:Company {name: "Company B"})
MERGE (a)-[:RELATIONSHIP_TYPE {
source: "URL",
confidence: CONFIDENCE_SCORE
}]->(b);

Example:

MERGE (a:Company {name: "Samsung Electronics"})
MERGE (b:Company {name: "NVIDIA"})
MERGE (a)-[:SUPPLIES {
source: "https://news.example.com/article",
confidence: 0.88
}]->(b);

---

# Tavily Search Usage

Use Tavily search queries focused on discovering company relationships.

Example search queries:

"company A supplier company B"
"company A partnership with company B"
"company A invests in company B"
"company A competitor company B"
"company A supplies chips to company B"

Search broadly and verify relationships using reliable sources.

---

# Extraction Rules

Only extract relationships that are clearly stated in the source.

Do NOT infer relationships.

Do NOT fabricate relationships.

If a relationship is uncertain, do not include it.

Prefer relationships mentioned in reputable sources such as:

* Reuters
* Bloomberg
* Financial Times
* TechCrunch
* official company press releases

---

# Graph Construction Rules

1. Avoid duplicate edges.
2. Always use MERGE instead of CREATE.
3. Standardize company names in English.
4. If multiple sources confirm the same relationship, include the most reliable source.
5. Each relationship must produce one Cypher statement.

---

# Output Rules

Your output must contain ONLY Cypher queries.

Do not include explanations.

Do not include markdown.

Do not include commentary.

Only return executable Cypher statements.

---

# Example Output

MERGE (a:Company {name: "TSMC"})
MERGE (b:Company {name: "Apple"})
MERGE (b)-[:CUSTOMER_OF {
source: "https://www.reuters.com/example",
confidence: 0.92
}]->(a);

MERGE (a:Company {name: "Samsung Electronics"})
MERGE (b:Company {name: "NVIDIA"})
MERGE (a)-[:SUPPLIES {
source: "https://www.bloomberg.com/example",
confidence: 0.85
}]->(b);

---

# Goal

Continuously build a reliable company relationship graph using verified web sources.


        """,
    name='web_agent',
)

In [29]:
from langchain.messages import HumanMessage

# Test the web agent
inputs = {
    "messages": [
        HumanMessage(
            content='Please output relationships with Samsung Electronics and NVIDIA'
        )
    ]
}

# Stream the web agent's response
for s in web_agent.stream(inputs, stream_mode="values"):
    message = s["messages"][-1]
    if isinstance(message, tuple):
        print(message)
    else:
        message.pretty_print()


================================ Human Message =================================

Please output relationships with Samsung Electronics and NVIDIA
================================== Ai Message ==================================
Name: web_agent
Tool Calls:
  tavily_search (call_fcmBj2HYtpKyQATlf8oxyBxr)
 Call ID: call_fcmBj2HYtpKyQATlf8oxyBxr
  Args:
    query: Samsung Electronics NVIDIA partnership
    start_date: None
    end_date: None
================================= Tool Message =================================
Name: tavily_search

{"query": "Samsung Electronics NVIDIA partnership", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://manufacturingdigital.com/news/nvidia-samsung-boost-ai-driven-semiconductor-manufacturing", "title": "Nvidia and Samsung's Boost for Semiconductor Manufacturing", "content": "Nvidia and Samsung's latest partnership involves the development of a new semiconductor AI factory powered by more than 50,000 Nvidia GPUs. Nvi

In [30]:
from IPython.display import Markdown

Markdown(message.content)

MERGE (a:Company {name: "Samsung Electronics"})
MERGE (b:Company {name: "NVIDIA"})
MERGE (a)-[:PARTNERS_WITH {
source: "https://news.samsung.com/global/samsung-teams-with-nvidia-to-lead-the-transformation-of-global-intelligent-manufacturing-through-new-ai-megafactory",
confidence: 0.92
}]->(b);

MERGE (a:Company {name: "NVIDIA"})
MERGE (b:Company {name: "Samsung Electronics"})
MERGE (a)-[:SUPPLIES {
source: "https://news.samsung.com/global/samsung-teams-with-nvidia-to-lead-the-transformation-of-global-intelligent-manufacturing-through-new-ai-megafactory",
confidence: 0.92
}]->(b);